# Distance From Home

## Objective

Build one row per World Cup team for 2010, 2014, 2018, 2022, and 2026 with the distance between the team's country and the World Cup host country. I use the same OpenStreetMap geocoding approach from the weather notebook through `tidygeocoder::geocode(method = "osm")`.

The 2026 team list comes from FIFA's qualified teams page: https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/articles/world-cup-2026-who-has-qualified

## Inputs

- `1.DataCleaning-R/Data/RDS/FullRoster.rds`

## Output

- `1.DataCleaning-R/Data/RDS/DistanceFromHome.rds`


In [1]:
library(tidyverse)
library(tidygeocoder)
library(here)
library(worldcup)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /Users/eialnisman/Desktop/WC2026Forecast



## Teams

The 2010-2022 teams come from the cleaned roster data. The 2026 teams are entered manually because that tournament is not in the current `worldcup` package data.

In [2]:
historical_teams <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullRoster.rds")) %>%
    distinct(tournament_id, team_name, team_id, team_code) %>%
    filter(tournament_id %in% c("WC-2010", "WC-2014", "WC-2018", "WC-2022"))

teams_2026 <- tribble(
    ~team_name, ~team_code,
    "Canada", "CAN",
    "Mexico", "MEX",
    "United States", "USA",
    "Australia", "AUS",
    "Iraq", "IRQ",
    "Iran", "IRN",
    "Japan", "JPN",
    "Jordan", "JOR",
    "South Korea", "KOR",
    "Qatar", "QAT",
    "Saudi Arabia", "SAU",
    "Uzbekistan", "UZB",
    "Algeria", "DZA",
    "Cabo Verde", "CPV",
    "Congo DR", "COD",
    "Ivory Coast", "CIV",
    "Egypt", "EGY",
    "Ghana", "GHA",
    "Morocco", "MAR",
    "Senegal", "SEN",
    "South Africa", "ZAF",
    "Tunisia", "TUN",
    "Curacao", "CUW",
    "Haiti", "HTI",
    "Panama", "PAN",
    "Argentina", "ARG",
    "Brazil", "BRA",
    "Colombia", "COL",
    "Ecuador", "ECU",
    "Paraguay", "PRY",
    "Uruguay", "URY",
    "New Zealand", "NZL",
    "Austria", "AUT",
    "Belgium", "BEL",
    "Bosnia and Herzegovina", "BIH",
    "Croatia", "HRV",
    "Czechia", "CZE",
    "England", "ENG",
    "France", "FRA",
    "Germany", "DEU",
    "Netherlands", "NLD",
    "Norway", "NOR",
    "Portugal", "PRT",
    "Scotland", "SCO",
    "Spain", "ESP",
    "Sweden", "SWE",
    "Switzerland", "CHE",
    "Turkey", "TUR"
) %>%
    mutate(
        tournament_id = "WC-2026",
        package_team_name = case_when(
            team_name == "Czechia" ~ "Czech Republic",
            TRUE ~ team_name
        )
    ) %>%
    left_join(
        worldcup::teams %>% select(package_team_name = team_name, team_id),
        by = "package_team_name"
    ) %>%
    select(tournament_id, team_name, team_id, team_code)

world_cup_teams <- bind_rows(historical_teams, teams_2026) %>%
    arrange(tournament_id, team_name)

world_cup_teams %>% count(tournament_id)

tournament_id,n
<chr>,<int>
WC-2010,32
WC-2014,32
WC-2018,32
WC-2022,32
WC-2026,48


## Geocode Countries

A few football names need clearer country names for geocoding.

In [3]:
geocode_name <- function(country) {
    case_when(
        country == "Congo DR" ~ "Democratic Republic of the Congo",
        country == "Curacao" ~ "Curacao",
        country == "England" ~ "England, United Kingdom",
        country == "Iran" ~ "Iran",
        country == "Ivory Coast" ~ "Cote d'Ivoire",
        country == "Scotland" ~ "Scotland, United Kingdom",
        country == "South Korea" ~ "South Korea",
        country == "United States" ~ "United States of America",
        TRUE ~ country
    )
}

country_locations <- world_cup_teams %>%
    distinct(team_name) %>%
    mutate(geocode_query = geocode_name(team_name)) %>%
    geocode(geocode_query, method = "osm", lat = country_lat, long = country_long)

host_locations <- tribble(
    ~tournament_id, ~host_country, ~host_geocode_query,
    "WC-2010", "South Africa", "South Africa",
    "WC-2014", "Brazil", "Brazil",
    "WC-2018", "Russia", "Russia",
    "WC-2022", "Qatar", "Qatar",
    "WC-2026", "United States", "United States of America"
) %>%
    geocode(host_geocode_query, method = "osm", lat = host_lat, long = host_long)

Passing 65 addresses to the Nominatim single address geocoder

Query completed in: 65.7 seconds

Passing 5 addresses to the Nominatim single address geocoder

Query completed in: 5 seconds



## Calculate Distance

Distance is great-circle distance in kilometers. Host countries get 0. For 2026, Canada and Mexico also get 0 because they are co-hosts.

In [4]:
haversine_km <- function(lat1, lon1, lat2, lon2) {
    earth_radius_km <- 6371
    to_radians <- function(degrees) degrees * pi / 180

    dlat <- to_radians(lat2 - lat1)
    dlon <- to_radians(lon2 - lon1)
    lat1 <- to_radians(lat1)
    lat2 <- to_radians(lat2)

    a <- sin(dlat / 2)^2 + cos(lat1) * cos(lat2) * sin(dlon / 2)^2
    2 * earth_radius_km * atan2(sqrt(a), sqrt(1 - a))
}

home_countries <- tribble(
    ~tournament_id, ~home_country,
    "WC-2010", "South Africa",
    "WC-2014", "Brazil",
    "WC-2018", "Russia",
    "WC-2022", "Qatar",
    "WC-2026", "United States",
    "WC-2026", "Mexico",
    "WC-2026", "Canada"
)

distance_from_home <- world_cup_teams %>%
    left_join(country_locations, by = "team_name") %>%
    left_join(host_locations, by = "tournament_id") %>%
    left_join(
        home_countries %>% mutate(is_home_country = TRUE),
        by = c("tournament_id", "team_name" = "home_country")
    ) %>%
    mutate(
        is_home_country = replace_na(is_home_country, FALSE),
        distance_from_host_km = if_else(
            is_home_country,
            0,
            haversine_km(country_lat, country_long, host_lat, host_long)
        )
    ) %>%
    select(
        tournament_id,
        team_name,
        team_id,
        team_code,
        host_country,
        is_home_country,
        country_lat,
        country_long,
        host_lat,
        host_long,
        distance_from_host_km
    ) %>%
    arrange(tournament_id, team_name)

distance_from_home %>%
    group_by(tournament_id) %>%
    summarize(
        teams = n(),
        missing_country_coordinates = sum(is.na(country_lat) | is.na(country_long)),
        missing_host_coordinates = sum(is.na(host_lat) | is.na(host_long)),
        home_country_rows = sum(is_home_country),
        .groups = "drop"
    )

tournament_id,teams,missing_country_coordinates,missing_host_coordinates,home_country_rows
<chr>,<int>,<int>,<int>,<int>
WC-2010,32,0,0,1
WC-2014,32,0,0,1
WC-2018,32,0,0,1
WC-2022,32,0,0,1
WC-2026,48,0,0,3


Inspect

In [6]:
distance_from_home

tournament_id,team_name,team_id,team_code,host_country,is_home_country,country_lat,country_long,host_lat,host_long,distance_from_host_km
<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WC-2010,Algeria,T-01,DZA,South Africa,FALSE,36.772933,3.058845,-28.81662,24.99164,7644.403
WC-2010,Argentina,T-03,ARG,South Africa,FALSE,-34.996496,-64.967282,-28.81662,24.99164,8219.664
WC-2010,Australia,T-04,AUS,South Africa,FALSE,-24.776109,134.755000,-28.81662,24.99164,10434.699
WC-2010,Brazil,T-09,BRA,South Africa,FALSE,-10.333333,-53.200000,-28.81662,24.99164,8313.019
WC-2010,Cameroon,T-11,CMR,South Africa,FALSE,4.612552,13.153581,-28.81662,24.99164,3926.776
WC-2010,Chile,T-13,CHL,South Africa,FALSE,-31.761336,-71.318770,-28.81662,24.99164,8907.301
WC-2010,Denmark,T-22,DNK,South Africa,FALSE,55.670249,10.333328,-28.81662,24.99164,9497.374
WC-2010,England,T-28,ENG,South Africa,FALSE,52.531021,-1.264906,-28.81662,24.99164,9398.536
WC-2010,France,T-30,FRA,South Africa,FALSE,46.603354,1.888334,-28.81662,24.99164,8702.228


## Save

In [7]:
saveRDS(distance_from_home, here("1.DataCleaning-R", "Data", "RDS", "DistanceFromHome.rds"))